In [3]:
import numpy as np  
import pandas as pd

from sklearn import set_config
set_config(display="text")

from sklearn.naive_bayes import BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics import accuracy_score

# 문제 정의

##### 베르누이 나이브베이즈 분류 모델을 사용하여 스팸 메일을 분류.

# 데이터 수집

##### 이번 실습에서는 간단한 스팸 메일 분류 실습을 위해 아래 이메일 타이틀과 스팸 여부가 있는 데이터를 사용.

In [10]:
email_list = [
                {'email title': 'free game only today', 'spam': True},
                {'email title': 'cheapest flight deal', 'spam': True},
                {'email title': 'limited time offer only today only today', 'spam': True},
                {'email title': 'today meeting schedule', 'spam': False},
                {'email title': 'your flight schedule attached', 'spam': False},
                {'email title': 'your credit card statement', 'spam': False}
             ]

df = pd.DataFrame(email_list)

df

,email title,spam
0,free game only today,True
1,cheapest flight deal,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


# 데이터 다듬기

##### sklearn의 베르누이 나이브베이즈 분류기는 숫자만을 다루기 때문에, True와 False를 1과 0으로 치환.

In [12]:
df['label'] = df['spam'].map({True:1, False:0})

df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deal,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


In [ ]:
# 학습에 사용될 데이터와 분류값을 나눔.
df_x = df['email title']
df_y = df['label']

In [ ]:
# 베르누이 나이브 베이즈 모델(Bernoulli Naïve Bayes)
cv = CountVectorizer(binary=True)
# email title 문자열 데이터 
# 문자열 데이터를 읽어 열거 등장하면 카운터 = +1 
# 알파벳 순으로 정리 -> 모든 단어들이 열거 있으면 1로 단어별로 그 위치에 1로 셋팅 나머지 0으로 셋팅 
x_traincv = cv.fit_transform(df_x) # 숫자형 형태로 변환

x_traincv # 훈련 데이터 6행 17개의 단어가 존재

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>

In [ ]:
cv.get_feature_names_out(df_x) # 17개의 단어 확인

array(['attached', 'card', 'cheapest', 'credit', 'deal', 'flight', 'free',
       'game', 'limited', 'meeting', 'offer', 'only', 'schedule',
       'statement', 'time', 'today', 'your'], dtype=object)

In [ ]:
encoded_input = x_traincv.toarray()
encoded_input

# 데이터 전처리 과정
# 머신러닝 알고리즘 문자열 데이터는 숫자형으로 변환하지 않는이상 동작하지 않음

array([[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
       [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

In [22]:
for i in range(6) :
  print(cv.inverse_transform(encoded_input[[i]])) # 리스트형태로 접근

[array(['free', 'game', 'only', 'today'], dtype='<U9')]
[array(['cheapest', 'deal', 'flight'], dtype='<U9')]
[array(['limited', 'offer', 'only', 'time', 'today'], dtype='<U9')]
[array(['meeting', 'schedule', 'today'], dtype='<U9')]
[array(['attached', 'flight', 'schedule', 'your'], dtype='<U9')]
[array(['card', 'credit', 'statement', 'your'], dtype='<U9')]


## 베르누이 나이브베이즈 분류 모델 생성

In [24]:
# 학습 데이터로 베르누이 분류기를 학습.
bnb = BernoulliNB()
y_train = df_y.astype('int') # 형변환

bnb.fit(x_traincv, y_train) # fit 학습

BernoulliNB()

In [28]:
# 테스트 데이터 다듬기
test_email_list = [
                {'email title': 'free flight offer', 'spam': True},
                {'email title': 'hey traveler free flight deal', 'spam': True},
                {'email title': 'limited free game offer', 'spam': True},
                {'email title': 'today flight schedule', 'spam': False},
                {'email title': 'your credit card attached', 'spam': False},
                {'email title': 'free credit card offer only today', 'spam': False}
             ]

test_df = pd.DataFrame(test_email_list)

test_df['label'] = test_df['spam'].map({True:1, False:0})

test_df

,email title,spam,label
0,free flight offer,True,1
1,hey traveler free flight deal,True,1
2,limited free game offer,True,1
3,today flight schedule,False,0
4,your credit card attached,False,0
5,free credit card offer only today,False,0


In [39]:
test_x = test_df['email title']
test_y = test_df['label'].astype('int')

In [ ]:
x_testcv = cv.transform(test_x) # 테스트 데이터는 새로운 기준 테이블을 만들면 안되서 윗단 fit을 제외

x_testcv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>

## 테스트

In [43]:
# 예측한 데이터 전달 -> 예측값을 반환 -> predicted 저장
predicted = bnb.predict(x_testcv) 

## 정확도

In [44]:
accuracy_score(test_y, predicted)

0.8333333333333334